# McLaren Race Intelligence Platform — Data Update

This notebook keeps the lab's F1 dataset current. It finds the latest round already
in GCS, fetches any newer rounds from the Jolpica API, and overwrites the affected
Parquet files in GCS. Students loading data in the lab will automatically get the
latest available data.

**Run this notebook before each lab event to freshen the data.**

### Tables updated

| Table | Source | Notes |
|-------|--------|-------|
| `races` | Jolpica | New race entries |
| `results` | Jolpica | Race results |
| `qualifying` | Jolpica | Qualifying results |
| `sprint_results` | Jolpica | Sprint race results |
| `driver_standings` | Jolpica | Round-by-round championship standings |
| `constructor_standings` | Jolpica | Round-by-round constructor standings |
| `constructor_results` | Derived | Per-constructor points per race, rebuilt from results |
| `drivers` | Jolpica | New entrants added automatically |
| `constructors` | Jolpica | New entrants added automatically |
| `circuits` | Jolpica | Full overwrite from circuits endpoint |
| `seasons` | Derived | New season year entries added |
| `mclaren_drivers` | Derived | Career stats rebuilt from updated results |
| `mclaren_profiles` | Gemini | Driver/team profiles regenerated for new or changed drivers |
| `mclaren_profiles_with_embeddings` | Gemini | Profiles with 3072-dim embeddings |

### Tables NOT updated

| Table | Reason |
|-------|--------|
| `lap_times` | Jolpica does not expose lap-by-lap timing data |
| `pit_stops` | Jolpica does not expose pit stop data |

These two tables retain Kaggle coverage through late 2024. The Ergast API
(which Jolpica mirrors) never included this data in its public endpoints.

**Requirements:** Colab Enterprise (IAM auth automatic), write access to `gs://class-demo`

---
## Section 0: Setup

In [1]:
!pip install -q google-cloud-storage pandas pyarrow requests google-genai tqdm

In [2]:
import google.auth
import pandas as pd
import numpy as np
import requests
import datetime
import time
import io
import json
import threading
from pathlib import Path
from google.cloud import storage
from google import genai
from google.genai import types

credentials, PROJECT_ID = google.auth.default()

# ── GCS constants ─────────────────────────────────────────────────────────
GCS_BUCKET     = "class-demo"
GCS_PREFIX     = "mclaren-f1"
GCS_BQ_STAGING = f"{GCS_PREFIX}/bq-staging"
GCS_PROF_PREFIX = f"{GCS_PREFIX}/profiles"

# ── API constants ─────────────────────────────────────────────────────────
MCLAREN_REF    = "mclaren"
JOLPICA_BASE   = "https://api.jolpi.ca/ergast/f1"
RATE_SLEEP     = 3.0
RETRY_BACKOFF  = 30.0
MAX_RETRIES    = 3

# ── Gemini constants ──────────────────────────────────────────────────────
GEMINI_MODEL   = "gemini-2.5-flash"
EMBED_MODEL    = "gemini-embedding-001"
EMBED_DIM      = 3072

gcs_client = storage.Client(project=PROJECT_ID)

print(f"✅ Project  : {PROJECT_ID}")
print(f"✅ GCS      : gs://{GCS_BUCKET}/{GCS_BQ_STAGING}/")
print(f"✅ Profiles : gs://{GCS_BUCKET}/{GCS_PROF_PREFIX}/")
print(f"✅ Gemini   : {GEMINI_MODEL} / {EMBED_MODEL} ({EMBED_DIM}-dim)")
print(f"✅ Run date : {datetime.date.today()}")


def fetch_json(url, retries=MAX_RETRIES):
    """Fetch a Jolpica URL with 429-aware backoff."""
    for attempt in range(retries):
        try:
            resp = requests.get(url, timeout=30)
            if resp.status_code == 200:
                return resp.json()
            elif resp.status_code == 429:
                wait = RETRY_BACKOFF * (attempt + 1)
                print(f"  429 — waiting {wait:.0f}s...")
                time.sleep(wait)
            else:
                print(f"  HTTP {resp.status_code} for {url}")
                return None
        except Exception as e:
            print(f"  Request error: {e}")
            time.sleep(RATE_SLEEP * 2)
    print(f"  ⚠️  Failed after {retries} attempts: {url}")
    return None

✅ Project  : qwiklabs-gcp-02-b36a0d0ed605
✅ GCS      : gs://class-demo/mclaren-f1/bq-staging/
✅ Profiles : gs://class-demo/mclaren-f1/profiles/
✅ Gemini   : gemini-2.5-flash / gemini-embedding-001 (3072-dim)
✅ Run date : 2026-04-03


In [3]:
# ── Gemini client (thread-safe for parallel profile generation) ────────────
_thread_local = threading.local()

def get_gemini_client():
    """Return a thread-local Gemini client."""
    if getattr(_thread_local, "client", None) is None:
        _thread_local.client = genai.Client(
            vertexai=True,
            project=PROJECT_ID,
            location="us-central1",
        )
    return _thread_local.client

# Quick connectivity test
test_resp = get_gemini_client().models.generate_content(
    model=GEMINI_MODEL,
    contents="In one sentence, what is McLaren Racing?",
    config=types.GenerateContentConfig(max_output_tokens=100)
)
print(f"✅ Gemini client ready")
print(f"   Test: {test_resp.text.strip()}")

✅ Gemini client ready
   Test: McLaren Racing is


---
## Section 1: Load Current Parquet Files from GCS

In [4]:
TABLES_TO_LOAD = [
    "races", "results", "qualifying", "sprint_results",
    "driver_standings", "constructor_standings", "constructor_results",
    "drivers", "constructors", "circuits", "seasons", "status",
    "mclaren_drivers",
]

bucket = gcs_client.bucket(GCS_BUCKET)
tables = {}

print("Loading tables from GCS...\n")
for name in TABLES_TO_LOAD:
    blob_path = f"{GCS_BQ_STAGING}/{name}.parquet"
    blob = bucket.blob(blob_path)
    if blob.exists():
        data = blob.download_as_bytes()
        tables[name] = pd.read_parquet(io.BytesIO(data))
        print(f"  ✅ {name:<30} {len(tables[name]):>8,} rows")
    else:
        print(f"  ❌ {name:<30} NOT FOUND")

missing = [t for t in TABLES_TO_LOAD if t not in tables]
if missing:
    raise RuntimeError(f"Cannot continue — missing tables: {missing}")

print(f"\n✅ All {len(tables)} tables loaded")

Loading tables from GCS...

  ✅ races                             1,173 rows
  ✅ results                          27,213 rows
  ✅ qualifying                       11,036 rows
  ✅ sprint_results                      502 rows
  ✅ driver_standings                 35,427 rows
  ✅ constructor_standings            13,664 rows
  ✅ constructor_results              13,298 rows
  ✅ drivers                             865 rows
  ✅ constructors                        214 rows
  ✅ circuits                             78 rows
  ✅ seasons                              77 rows
  ✅ status                              141 rows
  ✅ mclaren_drivers                      55 rows

✅ All 13 tables loaded


---
## Section 1.5: Load Existing Profiles from GCS

Load the current profile JSON so we can detect which drivers need new or
updated profiles after the race data update.

In [5]:
bucket = gcs_client.bucket(GCS_BUCKET)
existing_profiles = {}   # driver_id → profile dict

blob = bucket.blob(f"{GCS_PROF_PREFIX}/mclaren_profiles_with_embeddings.json")
if blob.exists():
    raw = json.loads(blob.download_as_string())
    for p in raw:
        existing_profiles[int(p["driver_id"])] = p
    print(f"✅ Loaded {len(existing_profiles)} existing profiles from GCS")
    # Show a quick summary
    has_embedding = sum(1 for p in existing_profiles.values() if p.get("embedding"))
    print(f"   With embeddings: {has_embedding}")
else:
    print("⚠️  No existing profiles found in GCS — all profiles will be generated fresh")

✅ Loaded 56 existing profiles from GCS
   With embeddings: 56


---
## Section 2: Find Latest Round and Determine What to Fetch

In [6]:
# Use results as source of truth for latest completed round
results_with_round = tables["results"].merge(
    tables["races"][["race_id", "year", "round"]], on="race_id", how="left"
)
latest_year  = int(results_with_round["year"].max())
latest_round = int(
    results_with_round[results_with_round["year"] == latest_year]["round"].max()
)
print(f"Latest data in GCS : {latest_year} Round {latest_round}")

current_year   = datetime.date.today().year
years_to_check = sorted(set([latest_year, current_year, current_year + 1]))
print(f"Checking years     : {years_to_check}")

rounds_needed = []  # (year, round, race_name)

for year in years_to_check:
    data = fetch_json(f"{JOLPICA_BASE}/{year}?limit=100")
    time.sleep(RATE_SLEEP)
    if not data:
        continue
    try:
        race_list = data["MRData"]["RaceTable"]["Races"]
    except (KeyError, TypeError):
        continue
    for race in race_list:
        rnd = int(race["round"])
        if year < latest_year:
            continue
        if year == latest_year and rnd <= latest_round:
            continue
        race_date = race.get("date", "")
        if race_date:
            try:
                if datetime.date.fromisoformat(race_date) > datetime.date.today():
                    continue
            except ValueError:
                pass
        rounds_needed.append((year, rnd, race.get("raceName", f"Round {rnd}")))

if rounds_needed:
    print(f"\n{len(rounds_needed)} new round(s) to fetch:")
    for y, r, name in rounds_needed:
        print(f"  {y} Round {r:2d} — {name}")
else:
    print("\n✅ Data is already up to date — nothing to fetch.")

Latest data in GCS : 2026 Round 3
Checking years     : [2026, 2027]

✅ Data is already up to date — nothing to fetch.


---
## Section 3: Refresh Circuits and Seasons

Circuits is a full overwrite from Jolpica, preserving existing IDs.
Seasons adds any new year entries.

In [7]:
# ── Circuits ───────────────────────────────────────────────────────────────
print("Refreshing circuits...")
all_circuits = []
offset = 0
while True:
    data = fetch_json(f"{JOLPICA_BASE}/circuits?limit=100&offset={offset}")
    time.sleep(RATE_SLEEP)
    if not data:
        break
    batch = data["MRData"]["CircuitTable"]["Circuits"]
    all_circuits.extend(batch)
    total = int(data["MRData"].get("total", len(all_circuits)))
    offset += 100
    if offset >= total:
        break

if all_circuits:
    existing = tables["circuits"]
    ref_col  = "circuit_ref" if "circuit_ref" in existing.columns else "circuit_id"
    ref_to_id = dict(zip(existing[ref_col].astype(str), existing["circuit_id"]))
    next_cid  = int(existing["circuit_id"].max()) + 1

    rows = []
    for c in all_circuits:
        ref = c.get("circuitId", "")
        cid = ref_to_id.get(ref)
        if cid is None:
            cid = next_cid
            next_cid += 1
            print(f"  + New circuit: {c.get('circuitName', ref)}")
        loc = c.get("Location", {})
        rows.append({
            "circuit_id":  cid,
            "circuit_ref": ref,
            "name":        c.get("circuitName", ""),
            "location":    loc.get("locality", ""),
            "country":     loc.get("country", ""),
            "lat":         pd.to_numeric(loc.get("lat"),   errors="coerce"),
            "lng":         pd.to_numeric(loc.get("long"),  errors="coerce"),
            "alt":         pd.to_numeric(loc.get("alt"),   errors="coerce"),
            "url":         c.get("url", ""),
        })
    tables["circuits"] = pd.DataFrame(rows)
    print(f"  ✅ circuits: {len(tables['circuits'])} rows")
else:
    print("  ⚠️  Could not fetch circuits — keeping existing table")


# ── Seasons ────────────────────────────────────────────────────────────────
existing_years = set(tables["seasons"]["year"].astype(int).tolist())
new_season_rows = []
for year in years_to_check:
    if year not in existing_years:
        new_season_rows.append({"year": year, "url": ""})
        print(f"  + New season: {year}")
if new_season_rows:
    tables["seasons"] = pd.concat(
        [tables["seasons"], pd.DataFrame(new_season_rows)], ignore_index=True
    )
print(f"  ✅ seasons: {len(tables['seasons'])} rows")

Refreshing circuits...
  ✅ circuits: 78 rows
  ✅ seasons: 77 rows


---
## Section 4: Fetch New Race Rounds from Jolpica

In [8]:
if not rounds_needed:
    print("✅ Nothing to fetch — skipping.")
else:
    # Lookup dicts
    driver_ref_to_id      = dict(zip(tables["drivers"]["driver_ref"].astype(str),      tables["drivers"]["driver_id"]))
    constructor_ref_to_id = dict(zip(tables["constructors"]["constructor_ref"].astype(str), tables["constructors"]["constructor_id"]))
    circuit_ref_to_id     = dict(zip(tables["circuits"]["circuit_ref"].astype(str),     tables["circuits"]["circuit_id"]))

    sprint_pk = "sprint_result_id" if "sprint_result_id" in tables["sprint_results"].columns else "result_id"

    next_ids = {
        "race_id":                  int(tables["races"]["race_id"].max()) + 1,
        "result_id":                int(tables["results"]["result_id"].max()) + 1,
        "qualify_id":               int(tables["qualifying"]["qualify_id"].max()) + 1,
        "sprint_result_id":         int(tables["sprint_results"][
            "sprint_result_id" if "sprint_result_id" in tables["sprint_results"].columns
            else "result_id"
        ].max()) + 1,
        "driver_standings_id":      int(tables["driver_standings"]["driver_standings_id"].max()) + 1,
        "constructor_standings_id": int(tables["constructor_standings"]["constructor_standings_id"].max()) + 1,
        "driver_id":                int(tables["drivers"]["driver_id"].max()) + 1,
        "constructor_id":           int(tables["constructors"]["constructor_id"].max()) + 1,
    }

    def get_or_create_driver(d):
        ref = d.get("driverId", "")
        if ref in driver_ref_to_id:
            return driver_ref_to_id[ref]
        did = next_ids["driver_id"]
        next_ids["driver_id"] += 1
        tables["drivers"] = pd.concat([tables["drivers"], pd.DataFrame([{
            "driver_id": did, "driver_ref": ref, "number": pd.NA,
            "code": d.get("code",""), "forename": d.get("givenName",""),
            "surname": d.get("familyName",""), "dob": d.get("dateOfBirth",""),
            "nationality": d.get("nationality",""), "url": d.get("url",""),
        }])], ignore_index=True)
        driver_ref_to_id[ref] = did
        print(f"    + New driver: {d.get('givenName','')} {d.get('familyName','')} ({ref})")
        return did

    def get_or_create_constructor(c):
        ref = c.get("constructorId", "")
        if ref in constructor_ref_to_id:
            return constructor_ref_to_id[ref]
        cid = next_ids["constructor_id"]
        next_ids["constructor_id"] += 1
        tables["constructors"] = pd.concat([tables["constructors"], pd.DataFrame([{
            "constructor_id": cid, "constructor_ref": ref,
            "name": c.get("name", ref), "nationality": c.get("nationality",""),
            "url": c.get("url",""),
        }])], ignore_index=True)
        constructor_ref_to_id[ref] = cid
        print(f"    + New constructor: {c.get('name', ref)} ({ref})")
        return cid

    new_races, new_results, new_qualifying = [], [], []
    new_sprint, new_drv_std, new_con_std   = [], [], []

    for year, rnd, race_name in rounds_needed:
        print(f"\n  {year} Round {rnd:2d} — {race_name}")
        race_id = next_ids["race_id"]

        # Race entry
        d = fetch_json(f"{JOLPICA_BASE}/{year}/{rnd}?limit=1")
        time.sleep(RATE_SLEEP)
        if d:
            try:
                r = d["MRData"]["RaceTable"]["Races"][0]
                cref = r["Circuit"]["circuitId"]
                new_races.append({
                    "race_id": race_id, "year": year, "round": rnd,
                    "circuit_id": circuit_ref_to_id.get(cref, pd.NA),
                    "name": r.get("raceName", race_name),
                    "date": r.get("date",""), "time": r.get("time",""), "url": r.get("url",""),
                })
                next_ids["race_id"] += 1
            except (KeyError, IndexError): pass

        # Results
        d = fetch_json(f"{JOLPICA_BASE}/{year}/{rnd}/results?limit=30")
        time.sleep(RATE_SLEEP)
        if d:
            try:
                rl = d["MRData"]["RaceTable"]["Races"]
                if rl:
                    for e in rl[0].get("Results", []):
                        new_results.append({
                            "result_id": next_ids["result_id"], "race_id": race_id,
                            "driver_id": get_or_create_driver(e["Driver"]),
                            "constructor_id": get_or_create_constructor(e["Constructor"]),
                            "number": pd.to_numeric(e.get("number"), errors="coerce"),
                            "grid": int(e.get("grid", 0)),
                            "position": pd.to_numeric(e.get("position"), errors="coerce"),
                            "position_text": e.get("positionText",""),
                            "position_order": int(e.get("positionOrder", 0)),
                            "points": float(e.get("points", 0)),
                            "laps": int(e.get("laps", 0)),
                            "time": e.get("Time",{}).get("time","") if "Time" in e else "",
                            "milliseconds": pd.to_numeric(e.get("Time",{}).get("millis") if "Time" in e else None, errors="coerce"),
                            "fastest_lap": pd.to_numeric(e.get("FastestLap",{}).get("lap") if "FastestLap" in e else None, errors="coerce"),
                            "rank": pd.to_numeric(e.get("FastestLap",{}).get("rank") if "FastestLap" in e else None, errors="coerce"),
                            "fastest_lap_time": e.get("FastestLap",{}).get("Time",{}).get("time","") if "FastestLap" in e else "",
                            "fastest_lap_speed": pd.to_numeric(e.get("FastestLap",{}).get("AverageSpeed",{}).get("speed") if "FastestLap" in e else None, errors="coerce"),
                            "status_id": 1,
                        })
                        next_ids["result_id"] += 1
                    print(f"    results   : {len(rl[0].get('Results',[]))}")
            except (KeyError, IndexError): pass

        # Qualifying
        d = fetch_json(f"{JOLPICA_BASE}/{year}/{rnd}/qualifying?limit=30")
        time.sleep(RATE_SLEEP)
        if d:
            try:
                rl = d["MRData"]["RaceTable"]["Races"]
                if rl:
                    for e in rl[0].get("QualifyingResults", []):
                        new_qualifying.append({
                            "qualify_id": next_ids["qualify_id"], "race_id": race_id,
                            "driver_id": get_or_create_driver(e["Driver"]),
                            "constructor_id": get_or_create_constructor(e["Constructor"]),
                            "number": pd.to_numeric(e.get("number"), errors="coerce"),
                            "position": int(e.get("position", 0)),
                            "q1": e.get("Q1",""), "q2": e.get("Q2",""), "q3": e.get("Q3",""),
                        })
                        next_ids["qualify_id"] += 1
                    print(f"    qualifying : {len(rl[0].get('QualifyingResults',[]))}")
            except (KeyError, IndexError): pass

        # Sprint
        d = fetch_json(f"{JOLPICA_BASE}/{year}/{rnd}/sprint?limit=30")
        time.sleep(RATE_SLEEP)
        if d:
            try:
                rl = d["MRData"]["RaceTable"]["Races"]
                if rl and rl[0].get("SprintResults"):
                    for e in rl[0]["SprintResults"]:
                        new_sprint.append({
                            "sprint_result_id": next_ids["sprint_result_id"], "race_id": race_id,
                            "driver_id": get_or_create_driver(e["Driver"]),
                            "constructor_id": get_or_create_constructor(e["Constructor"]),
                            "number": pd.to_numeric(e.get("number"), errors="coerce"),
                            "grid": int(e.get("grid", 0)),
                            "position": pd.to_numeric(e.get("position"), errors="coerce"),
                            "position_text": e.get("positionText",""),
                            "position_order": int(e.get("positionOrder", 0)),
                            "points": float(e.get("points", 0)),
                            "laps": int(e.get("laps", 0)),
                            "time": e.get("Time",{}).get("time","") if "Time" in e else "",
                            "milliseconds": pd.to_numeric(e.get("Time",{}).get("millis") if "Time" in e else None, errors="coerce"),
                            "fastest_lap_time": e.get("FastestLap",{}).get("Time",{}).get("time","") if "FastestLap" in e else "",
                            "status_id": 1,
                        })
                        next_ids["sprint_result_id"] += 1
                    print(f"    sprint     : {len(rl[0]['SprintResults'])}")
            except (KeyError, IndexError): pass

        # Driver standings
        d = fetch_json(f"{JOLPICA_BASE}/{year}/{rnd}/driverStandings?limit=100")
        time.sleep(RATE_SLEEP)
        if d:
            try:
                sl = d["MRData"]["StandingsTable"]["StandingsLists"]
                if sl:
                    for e in sl[0].get("DriverStandings", []):
                        new_drv_std.append({
                            "driver_standings_id": next_ids["driver_standings_id"],
                            "race_id": race_id,
                            "driver_id": get_or_create_driver(e["Driver"]),
                            "points": float(e.get("points", 0)),
                            "position": int(e.get("position", 0)),
                            "position_text": e.get("positionText",""),
                            "wins": int(e.get("wins", 0)),
                        })
                        next_ids["driver_standings_id"] += 1
                    print(f"    drv_stnd   : {len(sl[0].get('DriverStandings',[]))}")
            except (KeyError, IndexError): pass

        # Constructor standings
        d = fetch_json(f"{JOLPICA_BASE}/{year}/{rnd}/constructorStandings?limit=100")
        time.sleep(RATE_SLEEP)
        if d:
            try:
                sl = d["MRData"]["StandingsTable"]["StandingsLists"]
                if sl:
                    for e in sl[0].get("ConstructorStandings", []):
                        new_con_std.append({
                            "constructor_standings_id": next_ids["constructor_standings_id"],
                            "race_id": race_id,
                            "constructor_id": get_or_create_constructor(e["Constructor"]),
                            "points": float(e.get("points", 0)),
                            "position": int(e.get("position", 0)),
                            "position_text": e.get("positionText",""),
                            "wins": int(e.get("wins", 0)),
                        })
                        next_ids["constructor_standings_id"] += 1
                    print(f"    con_stnd   : {len(sl[0].get('ConstructorStandings',[]))}")
            except (KeyError, IndexError): pass

    # Merge
    print("\n── Merging ───────────────────────────────────────────────────────────")

    def merge_rows(table_name, new_rows, dedup_keys):
        if not new_rows:
            print(f"  {table_name:<30} no new rows")
            return
        combined = pd.concat([tables[table_name], pd.DataFrame(new_rows)], ignore_index=True)
        valid_keys = [k for k in dedup_keys if k in combined.columns]
        if valid_keys:
            combined = combined.drop_duplicates(subset=valid_keys, keep="last")
        tables[table_name] = combined.reset_index(drop=True)
        print(f"  {table_name:<30} +{len(new_rows)} rows → {len(tables[table_name]):,} total")

    merge_rows("races",                new_races,    ["year", "round"])
    merge_rows("results",              new_results,  ["race_id", "driver_id"])
    merge_rows("qualifying",           new_qualifying,["race_id", "driver_id"])
    merge_rows("sprint_results",       new_sprint,   ["race_id", "driver_id"])
    merge_rows("driver_standings",     new_drv_std,  ["race_id", "driver_id"])
    merge_rows("constructor_standings",new_con_std,  ["race_id", "constructor_id"])
    print(f"  {'drivers':<30} {len(tables['drivers']):,} total")
    print(f"  {'constructors':<30} {len(tables['constructors']):,} total")
    print("\n✅ Race data merged")

✅ Nothing to fetch — skipping.


---
## Section 5: Rebuild Derived Tables

Rebuild `constructor_results` and `mclaren_drivers` from the updated results.
Always rebuilt from scratch to stay consistent with the source data.

In [9]:
# ── constructor_results ────────────────────────────────────────────────────
print("Rebuilding constructor_results...")
con_results = (
    tables["results"]
    .groupby(["race_id", "constructor_id"])
    .agg(points=("points", "sum"))
    .reset_index()
)
con_results["constructor_results_id"] = range(1, len(con_results) + 1)
con_results["status"] = "Finished"
tables["constructor_results"] = con_results
print(f"  ✅ constructor_results: {len(tables['constructor_results']):,} rows")


# ── mclaren_drivers ────────────────────────────────────────────────────────
print("\nRebuilding mclaren_drivers...")

mclaren_id = tables["constructors"][
    tables["constructors"]["constructor_ref"] == MCLAREN_REF
].iloc[0]["constructor_id"]

mac_results = tables["results"][tables["results"]["constructor_id"] == mclaren_id].copy()
mac_results = mac_results.merge(
    tables["races"][["race_id", "year", "round", "name", "circuit_id"]], on="race_id"
)

driver_stats = mac_results.groupby("driver_id").agg(
    mclaren_first_year=("year", "min"),
    mclaren_last_year=("year", "max"),
    mclaren_races=("race_id", "count"),
    mclaren_points=("points", "sum"),
).reset_index()

pos_col = "position_order" if "position_order" in mac_results.columns else "position"
pos_numeric = pd.to_numeric(mac_results[pos_col], errors="coerce")
mac_results["_pos"] = pos_numeric

wins_df   = mac_results[mac_results["_pos"] == 1].groupby("driver_id").size().reset_index(name="mclaren_wins")
podium_df = mac_results[mac_results["_pos"] <= 3].groupby("driver_id").size().reset_index(name="mclaren_podiums")

driver_stats = driver_stats.merge(wins_df,   on="driver_id", how="left")
driver_stats = driver_stats.merge(podium_df, on="driver_id", how="left")
driver_stats[["mclaren_wins", "mclaren_podiums"]] = \
    driver_stats[["mclaren_wins", "mclaren_podiums"]].fillna(0).astype(int)

mclaren_drivers = tables["drivers"][
    tables["drivers"]["driver_id"].isin(driver_stats["driver_id"])
].merge(driver_stats, on="driver_id")
mclaren_drivers = mclaren_drivers.sort_values("mclaren_first_year").reset_index(drop=True)

tables["mclaren_drivers"] = mclaren_drivers
print(f"  ✅ mclaren_drivers: {len(tables['mclaren_drivers'])} drivers")

# Spot-check current drivers
print("\n  Current/recent McLaren drivers:")
recent_drivers = mclaren_drivers[
    mclaren_drivers["mclaren_last_year"] >= datetime.date.today().year - 1
].sort_values("mclaren_last_year", ascending=False)
for _, row in recent_drivers.head(6).iterrows():
    print(f"    {row['forename']} {row['surname']:<20} "
          f"{int(row['mclaren_first_year'])}–{int(row['mclaren_last_year'])}  "
          f"{int(row['mclaren_races'])} races  {int(row['mclaren_wins'])} wins")

Rebuilding constructor_results...
  ✅ constructor_results: 13,298 rows

Rebuilding mclaren_drivers...
  ✅ mclaren_drivers: 55 drivers

  Current/recent McLaren drivers:
    Lando Norris               2019–2026  155 races  11 wins
    Oscar Piastri              2023–2026  73 races  9 wins


---
## Section 5.5: Update McLaren Driver & Team Profiles

Compare the refreshed `mclaren_drivers` stats against existing profiles to
determine which drivers need a new or regenerated profile. A profile is
regenerated if:

1. **New driver** — no existing profile for this `driver_id`
2. **Stats changed** — wins, podiums, races, or last season changed since the
   profile was generated (meaning the narrative is stale)

Profiles that haven't changed are carried forward as-is, preserving their
existing embeddings. Only changed profiles get new Gemini generation +
embedding calls, keeping API usage minimal for routine updates.

The team (constructor) profile is always regenerated since aggregate stats
change with every new round.

In [10]:
# ── Determine which drivers need profile updates ─────────────────────────
drivers_needing_update = []
drivers_carried_forward = []

for _, row in tables["mclaren_drivers"].iterrows():
    did = int(row["driver_id"])
    existing = existing_profiles.get(did)

    if existing is None:
        # Brand-new McLaren driver — no profile exists
        drivers_needing_update.append(did)
        continue

    # Compare key stats that would change the narrative
    meta = existing.get("metadata", {})
    changed = False
    for key, col in [
        ("mclaren_wins",    "mclaren_wins"),
        ("mclaren_podiums", "mclaren_podiums"),
        ("mclaren_races",   "mclaren_races"),
        ("mclaren_points",  "mclaren_points"),
        ("seasons_at_mclaren", None),  # handled separately
    ]:
        if col and int(meta.get(key, -1)) != int(row[col]):
            changed = True
            break

    # Check if last season changed (driver raced in a new year)
    existing_last = meta.get("seasons_at_mclaren", [])
    if existing_last and int(existing_last[-1]) != int(row["mclaren_last_year"]):
        changed = True

    if changed:
        drivers_needing_update.append(did)
    else:
        drivers_carried_forward.append(did)

print(f"McLaren drivers total : {len(tables['mclaren_drivers'])}")
print(f"Profiles to generate  : {len(drivers_needing_update)}")
print(f"Profiles carried over : {len(drivers_carried_forward)}")

if drivers_needing_update:
    print("\nDrivers needing new profiles:")
    for did in drivers_needing_update:
        drv = tables["mclaren_drivers"][tables["mclaren_drivers"]["driver_id"] == did].iloc[0]
        existing_tag = "" if did not in existing_profiles else " (stats changed)"
        if did not in existing_profiles:
            existing_tag = " (NEW)"
        print(f"  {drv['forename']} {drv['surname']}{existing_tag}")

McLaren drivers total : 55
Profiles to generate  : 2
Profiles carried over : 53

Drivers needing new profiles:
  Lando Norris (stats changed)
  Oscar Piastri (stats changed)


In [11]:
# ── Profile generation functions ───────────────────────────────────────────
# Same prompt design and generation logic as the data_prep notebook.

def build_driver_context(driver_row, mac_results_df, races_df):
    """Build a structured context dict for a McLaren driver."""
    did = driver_row["driver_id"]
    full_name = f"{driver_row['forename']} {driver_row['surname']}"

    drv_results = mac_results_df[mac_results_df["driver_id"] == did].copy()
    drv_results = drv_results.merge(races_df[["race_id", "year", "name", "round"]], on="race_id")

    seasons = sorted(drv_results["year"].unique().tolist())
    pos_col = "position_order" if "position_order" in drv_results.columns else "position"
    pos_num = pd.to_numeric(drv_results.get(pos_col, pd.Series(dtype=float)), errors="coerce")

    wins    = int((pos_num == 1).sum())
    podiums = int((pos_num <= 3).sum())
    points  = float(drv_results["points"].sum()) if "points" in drv_results else 0.0

    top = drv_results.copy()
    top["_pos"] = pos_num
    top = top[top["_pos"] <= 10].sort_values(["_pos", "year"]).head(8)
    notable = [
        f"P{int(r['_pos'])} — {r['name']} {int(r['year'])}"
        for _, r in top.iterrows()
    ]

    return {
        "driver_id": int(did),
        "full_name": full_name,
        "forename": driver_row["forename"],
        "surname": driver_row["surname"],
        "nationality": driver_row.get("nationality", ""),
        "dob": str(driver_row.get("dob", "")),
        "seasons_at_mclaren": [int(y) for y in seasons],
        "mclaren_races": int(len(drv_results)),
        "mclaren_wins": wins,
        "mclaren_podiums": podiums,
        "mclaren_points": round(points, 1),
        "notable_results": notable,
    }


def build_profile_prompt(ctx):
    seasons_str = ", ".join(str(y) for y in ctx["seasons_at_mclaren"])
    notable_str = "\n".join(f"  - {r}" for r in ctx["notable_results"]) \
                  if ctx["notable_results"] else "  - No top-10 finishes in dataset"

    return f"""Write a factual 2–3 paragraph profile of **{ctx['full_name']}** focused \
specifically on their time racing for the McLaren Formula 1 team.

IMPORTANT CONSTRAINTS:
- Cover their McLaren chapter only. Mention their pre-McLaren career briefly (1–2 sentences) \
for context, but do NOT discuss what they did after leaving McLaren.
- Write in a neutral, encyclopedic tone suitable for a technical reference document.
- Be factual and specific. Include seasons, key races, championship results where relevant.
- Keep total length to 200–350 words.

KNOWN DATA FROM THE DATASET:
- Nationality: {ctx['nationality']}
- Seasons at McLaren: {seasons_str}
- Races with McLaren: {ctx['mclaren_races']}
- Wins with McLaren: {ctx['mclaren_wins']}
- Podiums with McLaren: {ctx['mclaren_podiums']}
- Points with McLaren: {ctx['mclaren_points']}
- Notable results (from race data):
{notable_str}

Use these data points as anchors. You may use search to fill in additional accurate details \
such as championship positions, specific race names, or career significance."""


def generate_driver_profile(ctx, max_retries=3, retry_delay=2.0):
    """Generate a profile for one driver. Returns (profile_text, error)."""
    prompt = build_profile_prompt(ctx)
    gen_config = types.GenerateContentConfig(
        temperature=0.7,
        tools=[types.Tool(google_search=types.GoogleSearch())],
    )
    for attempt in range(max_retries):
        try:
            resp = get_gemini_client().models.generate_content(
                model=GEMINI_MODEL,
                contents=prompt,
                config=gen_config,
            )
            text = resp.text.strip() if resp.text else None
            if text and len(text) > 80:
                return text, None
            if attempt < max_retries - 1:
                time.sleep(retry_delay)
        except Exception as e:
            if attempt < max_retries - 1:
                time.sleep(retry_delay * (attempt + 1))
            else:
                return None, f"{type(e).__name__}: {e}"
    return None, "Max retries exceeded or response too short"


def generate_embedding(text, label, max_retries=3, retry_delay=2.0):
    """Generate a 3072-dim embedding for a profile text string."""
    for attempt in range(max_retries):
        try:
            resp = get_gemini_client().models.embed_content(
                model=EMBED_MODEL,
                contents=text,
                config=types.EmbedContentConfig(output_dimensionality=EMBED_DIM)
            )
            if resp.embeddings and resp.embeddings[0].values:
                return list(resp.embeddings[0].values), None
            if attempt < max_retries - 1:
                time.sleep(retry_delay)
        except Exception as e:
            if attempt < max_retries - 1:
                time.sleep(retry_delay * (attempt + 1))
            else:
                return None, f"{type(e).__name__}: {e}"
    return None, "No embedding returned after retries"


print("✅ Profile generation functions defined")

✅ Profile generation functions defined


In [12]:
# ── Generate profiles for drivers that need updates ───────────────────────
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.notebook import tqdm

MAX_WORKERS = 20

# Build McLaren results subset for context building
mclaren_id = tables["constructors"][
    tables["constructors"]["constructor_ref"] == MCLAREN_REF
].iloc[0]["constructor_id"]

mac_results_for_ctx = tables["results"][
    tables["results"]["constructor_id"] == mclaren_id
].copy()
races_for_ctx = tables["races"][["race_id", "year", "name", "round"]]

# Build contexts for drivers needing updates
update_contexts = []
for did in drivers_needing_update:
    row = tables["mclaren_drivers"][tables["mclaren_drivers"]["driver_id"] == did].iloc[0]
    ctx = build_driver_context(row, mac_results_for_ctx, races_for_ctx)
    update_contexts.append(ctx)


def process_one_driver(ctx):
    """Worker: generate profile + embedding for one driver."""
    text, err = generate_driver_profile(ctx)
    result = {
        "driver_id": ctx["driver_id"],
        "full_name": ctx["full_name"],
        "metadata": {
            "nationality": ctx["nationality"],
            "dob": ctx["dob"],
            "seasons_at_mclaren": ctx["seasons_at_mclaren"],
            "mclaren_races": ctx["mclaren_races"],
            "mclaren_wins": ctx["mclaren_wins"],
            "mclaren_podiums": ctx["mclaren_podiums"],
            "mclaren_points": ctx["mclaren_points"],
        },
        "profile_text": text,
        "profile_error": err,
        "embedding": None,
        "embedding_error": None,
    }

    # Generate embedding if profile succeeded
    if text:
        vec, emb_err = generate_embedding(text, ctx["full_name"])
        result["embedding"] = vec
        result["embedding_error"] = emb_err

    return result


updated_profiles = {}   # driver_id → profile dict

if update_contexts:
    print(f"Generating {len(update_contexts)} driver profiles + embeddings...\n")
    with tqdm(total=len(update_contexts), desc="Driver profiles") as pbar:
        with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
            futures = {executor.submit(process_one_driver, ctx): ctx for ctx in update_contexts}
            for future in as_completed(futures):
                result = future.result()
                updated_profiles[result["driver_id"]] = result
                status = "✅" if result.get("profile_text") else "❌"
                pbar.set_postfix_str(f"{status} {result['full_name']}")
                pbar.update(1)

    errors = [p for p in updated_profiles.values() if p.get("profile_error")]
    embed_errors = [p for p in updated_profiles.values() if p.get("embedding_error")]
    print(f"\n✅ Profiles generated: {len(updated_profiles) - len(errors)}")
    if errors:
        for e in errors:
            print(f"  ❌ {e['full_name']}: {e['profile_error']}")
    if embed_errors:
        for e in embed_errors:
            print(f"  ⚠️  Embedding failed: {e['full_name']}: {e['embedding_error']}")
else:
    print("✅ No driver profiles need updating")

Generating 2 driver profiles + embeddings...



Driver profiles:   0%|          | 0/2 [00:00<?, ?it/s]


✅ Profiles generated: 2


In [15]:
# ── Regenerate team profile (always — aggregate stats change each round) ──
# Brief cooldown after the parallel driver profile burst to avoid 429s.
print("Pausing 15s after driver profile burst to reset rate limits...")
time.sleep(15)

print("Generating McLaren team profile...")

mac_con_standings = tables.get("constructor_standings", pd.DataFrame())
if not mac_con_standings.empty:
    mac_standings = mac_con_standings[
        mac_con_standings["constructor_id"] == mclaren_id
    ].merge(tables["races"][["race_id", "year"]], on="race_id", how="left")
    championship_years = mac_standings[mac_standings["position"] == 1]["year"].unique().tolist()
else:
    championship_years = []

total_mac_wins  = int((pd.to_numeric(mac_results_for_ctx.get(
    "position_order", mac_results_for_ctx.get("position", pd.Series())), errors="coerce") == 1).sum())
total_mac_races = mac_results_for_ctx["race_id"].nunique()

team_prompt = f"""Write a factual 3–4 paragraph profile of the **McLaren Formula 1 team**.

Cover:
1. Founding and early history (Bruce McLaren era)
2. Championship eras — including the dominant 1980s–90s period with Senna and Prost
3. Post-2000 performance through to the current season
4. The team's identity, culture, and engineering philosophy

KNOWN DATA FROM DATASET:
- Total race entries: {total_mac_races:,}
- Total race wins: {total_mac_wins:,}
- Constructor championship seasons (from data): {sorted(championship_years)}

Tone: neutral and encyclopedic, suitable for a technical reference document.
Length: 300–450 words.
Use search to verify specific championship years, driver rosters, and historical details."""

team_gen_config = types.GenerateContentConfig(
    temperature=0.6,
    tools=[types.Tool(google_search=types.GoogleSearch())],
)

# Retry with backoff (same pattern as driver profiles)
team_profile_text = None
for attempt in range(5):
    try:
        team_resp = get_gemini_client().models.generate_content(
            model=GEMINI_MODEL,
            contents=team_prompt,
            config=team_gen_config,
        )
        team_profile_text = team_resp.text.strip()
        if team_profile_text and len(team_profile_text) > 80:
            break
    except Exception as e:
        wait = 15 * (attempt + 1)
        print(f"  ⚠️  Attempt {attempt+1} failed ({type(e).__name__}) — waiting {wait}s...")
        time.sleep(wait)

if not team_profile_text:
    raise RuntimeError("Team profile generation failed after 5 attempts — try re-running this cell")

# Embed team profile
team_vec, team_emb_err = generate_embedding(team_profile_text, "McLaren Racing")

team_profile = {
    "driver_id": -1,
    "full_name": "McLaren Racing (Constructor)",
    "metadata": {"type": "constructor", "constructor_ref": MCLAREN_REF},
    "profile_text": team_profile_text,
    "profile_error": None,
    "embedding": team_vec,
    "embedding_error": team_emb_err,
}
updated_profiles[-1] = team_profile

print(f"✅ Team profile generated ({len(team_profile_text)} chars)")
if team_emb_err:
    print(f"  ⚠️  Embedding error: {team_emb_err}")
else:
    print(f"  ✅ Embedding: {len(team_vec)} dimensions")

Pausing 15s after driver profile burst to reset rate limits...
Generating McLaren team profile...
✅ Team profile generated (3212 chars)
  ✅ Embedding: 3072 dimensions


In [16]:
# ── Merge updated profiles with carried-forward profiles ──────────────────
final_profiles = []

# Carry forward unchanged profiles
for did in drivers_carried_forward:
    if did in existing_profiles:
        final_profiles.append(existing_profiles[did])

# Add newly generated / updated profiles
for did, profile in updated_profiles.items():
    final_profiles.append(profile)

# Sort by driver_id for deterministic output
final_profiles = sorted(final_profiles, key=lambda x: x["driver_id"])

profiles_with_text  = [p for p in final_profiles if p.get("profile_text")]
profiles_with_embed = [p for p in final_profiles if p.get("embedding")]

print(f"── Profile summary ────────────────────────────────────────────────")
print(f"  Total profiles        : {len(final_profiles)}")
print(f"  With text             : {len(profiles_with_text)}")
print(f"  With embeddings       : {len(profiles_with_embed)}")
print(f"  Newly generated       : {len(updated_profiles)}")
print(f"  Carried forward       : {len(drivers_carried_forward)}")

── Profile summary ────────────────────────────────────────────────
  Total profiles        : 56
  With text             : 56
  With embeddings       : 56
  Newly generated       : 3
  Carried forward       : 53


---
## Section 6: Write Updated Parquet Files and Profiles to GCS

Writes all updated race data tables as Parquet, plus four profile files:
- `mclaren_profiles.json` — profiles without embeddings (human-readable)
- `mclaren_profiles_with_embeddings.json` — full profiles with 3072-dim vectors
- `mclaren_profiles.parquet` — profiles without embeddings (for BigQuery / analytics)
- `mclaren_profiles_with_embeddings.parquet` — full profiles with embeddings as arrays

In [17]:
TABLES_TO_WRITE = [
    "races", "results", "qualifying", "sprint_results",
    "driver_standings", "constructor_standings", "constructor_results",
    "drivers", "constructors", "circuits", "seasons",
    "mclaren_drivers",
]

print(f"Writing {len(TABLES_TO_WRITE)} Parquet files to GCS...\n")
bucket   = gcs_client.bucket(GCS_BUCKET)
total_mb = 0

for name in TABLES_TO_WRITE:
    df  = tables[name]
    buf = io.BytesIO()
    df.to_parquet(buf, index=False)
    size_mb = buf.tell() / 1024 / 1024
    total_mb += size_mb
    buf.seek(0)
    blob_path = f"{GCS_BQ_STAGING}/{name}.parquet"
    bucket.blob(blob_path).upload_from_file(buf, content_type="application/octet-stream")
    print(f"  ✅ {name:<30} {len(df):>8,} rows  ({size_mb:.2f} MB)")

print(f"\n  Race data total : {total_mb:.1f} MB")
print(f"  Not updated     : lap_times, pit_stops (Jolpica API limitation)")

Writing 12 Parquet files to GCS...

  ✅ races                             1,173 rows  (0.04 MB)
  ✅ results                          27,213 rows  (0.57 MB)
  ✅ qualifying                       11,036 rows  (0.22 MB)
  ✅ sprint_results                      502 rows  (0.02 MB)
  ✅ driver_standings                 35,427 rows  (0.34 MB)
  ✅ constructor_standings            13,664 rows  (0.13 MB)
  ✅ constructor_results              13,298 rows  (0.10 MB)
  ✅ drivers                             865 rows  (0.05 MB)
  ✅ constructors                        214 rows  (0.01 MB)
  ✅ circuits                             78 rows  (0.01 MB)
  ✅ seasons                              77 rows  (0.00 MB)
  ✅ mclaren_drivers                      55 rows  (0.01 MB)

  Race data total : 1.5 MB
  Not updated     : lap_times, pit_stops (Jolpica API limitation)


In [18]:
# ── Write profile files (JSON + Parquet) ──────────────────────────────────
print("\nWriting profile files to GCS...\n")

bucket = gcs_client.bucket(GCS_BUCKET)
profile_total_mb = 0

# ── 1. JSON files ─────────────────────────────────────────────────────────
# Full profiles with embeddings
full_json = json.dumps(final_profiles, ensure_ascii=False)
blob = bucket.blob(f"{GCS_PROF_PREFIX}/mclaren_profiles_with_embeddings.json")
blob.upload_from_string(full_json, content_type="application/json")
size_mb = len(full_json.encode()) / 1024 / 1024
profile_total_mb += size_mb
print(f"  ✅ mclaren_profiles_with_embeddings.json    ({size_mb:.1f} MB)")

# Light profiles without embeddings
light = [{k: v for k, v in p.items() if k not in ("embedding", "embedding_error")}
         for p in final_profiles]
light_json = json.dumps(light, indent=2, ensure_ascii=False)
blob = bucket.blob(f"{GCS_PROF_PREFIX}/mclaren_profiles.json")
blob.upload_from_string(light_json, content_type="application/json")
size_mb = len(light_json.encode()) / 1024 / 1024
profile_total_mb += size_mb
print(f"  ✅ mclaren_profiles.json                    ({size_mb:.2f} MB)")

# ── 2. Parquet files ──────────────────────────────────────────────────────
# Build a DataFrame from profiles, flattening metadata into columns.
# Embeddings are stored as a list column (Parquet supports nested arrays).

def profiles_to_dataframe(profiles, include_embeddings=False):
    """Convert the profile list to a flat DataFrame suitable for Parquet."""
    rows = []
    for p in profiles:
        meta = p.get("metadata", {})
        row = {
            "driver_id":          p.get("driver_id"),
            "full_name":          p.get("full_name", ""),
            "nationality":        meta.get("nationality", ""),
            "dob":                meta.get("dob", ""),
            "type":               meta.get("type", "driver"),
            "constructor_ref":    meta.get("constructor_ref", ""),
            "seasons_at_mclaren": json.dumps(meta.get("seasons_at_mclaren", [])),
            "mclaren_races":      meta.get("mclaren_races", 0),
            "mclaren_wins":       meta.get("mclaren_wins", 0),
            "mclaren_podiums":    meta.get("mclaren_podiums", 0),
            "mclaren_points":     meta.get("mclaren_points", 0.0),
            "profile_text":       p.get("profile_text", ""),
        }
        if include_embeddings:
            row["embedding"] = p.get("embedding")
        rows.append(row)
    return pd.DataFrame(rows)


# Profiles without embeddings → Parquet
df_light = profiles_to_dataframe(final_profiles, include_embeddings=False)
buf = io.BytesIO()
df_light.to_parquet(buf, index=False)
size_mb = buf.tell() / 1024 / 1024
profile_total_mb += size_mb
buf.seek(0)
bucket.blob(f"{GCS_PROF_PREFIX}/mclaren_profiles.parquet").upload_from_file(
    buf, content_type="application/octet-stream"
)
print(f"  ✅ mclaren_profiles.parquet                 ({size_mb:.2f} MB)  [{len(df_light)} rows]")

# Profiles with embeddings → Parquet
df_full = profiles_to_dataframe(final_profiles, include_embeddings=True)
buf = io.BytesIO()
df_full.to_parquet(buf, index=False)
size_mb = buf.tell() / 1024 / 1024
profile_total_mb += size_mb
buf.seek(0)
bucket.blob(f"{GCS_PROF_PREFIX}/mclaren_profiles_with_embeddings.parquet").upload_from_file(
    buf, content_type="application/octet-stream"
)
print(f"  ✅ mclaren_profiles_with_embeddings.parquet  ({size_mb:.1f} MB)  [{len(df_full)} rows]")

# Also copy the light Parquet to bq-staging for easy BigQuery load alongside other tables
buf = io.BytesIO()
df_light.to_parquet(buf, index=False)
buf.seek(0)
bucket.blob(f"{GCS_BQ_STAGING}/mclaren_profiles.parquet").upload_from_file(
    buf, content_type="application/octet-stream"
)
buf = io.BytesIO()
df_full.to_parquet(buf, index=False)
buf.seek(0)
bucket.blob(f"{GCS_BQ_STAGING}/mclaren_profiles_with_embeddings.parquet").upload_from_file(
    buf, content_type="application/octet-stream"
)
print(f"  ✅ Copies also written to bq-staging/ for BigQuery load")

print(f"\n  Profile files total : {profile_total_mb:.1f} MB")
print(f"\n✅ All files written to GCS")


Writing profile files to GCS...

  ✅ mclaren_profiles_with_embeddings.json    (3.8 MB)
  ✅ mclaren_profiles.json                    (0.10 MB)
  ✅ mclaren_profiles.parquet                 (0.06 MB)  [56 rows]
  ✅ mclaren_profiles_with_embeddings.parquet  (1.4 MB)  [56 rows]
  ✅ Copies also written to bq-staging/ for BigQuery load

  Profile files total : 5.3 MB

✅ All files written to GCS


---
## Section 7: Verification

In [19]:
merged = tables["results"].merge(
    tables["races"][["race_id", "year", "round"]], on="race_id", how="left"
)
new_latest_year  = int(merged["year"].max())
new_latest_round = int(merged[merged["year"] == new_latest_year]["round"].max())

print("── Coverage (last 5 seasons) ─────────────────────────────────────────")
recent = merged[merged["year"] >= new_latest_year - 4]
summary = (
    recent.groupby("year")
    .agg(rounds=("round", "nunique"), result_rows=("result_id", "count"))
    .reset_index()
    .sort_values("year", ascending=False)
)
for _, row in summary.iterrows():
    print(f"  {int(row['year'])}: {int(row['rounds']):2d} rounds, {int(row['result_rows']):4d} result rows")

print(f"\n── Final counts ──────────────────────────────────────────────────────")
for name in TABLES_TO_WRITE:
    print(f"  {name:<30} {len(tables[name]):>8,}")
print(f"  {'lap_times (unchanged)':<30} {len(tables.get('lap_times', pd.DataFrame())):>8,}  (Kaggle only)")
print(f"  {'pit_stops (unchanged)':<30} {len(tables.get('pit_stops', pd.DataFrame())):>8,}  (Kaggle only)")

print(f"\n── Profile counts ────────────────────────────────────────────────────")
print(f"  Total profiles           : {len(final_profiles)}")
print(f"  With embeddings          : {len([p for p in final_profiles if p.get('embedding')])}")
print(f"  Regenerated this run     : {len(updated_profiles)}")

print(f"\n── GCS profile files ─────────────────────────────────────────────────")
for blob in gcs_client.list_blobs(GCS_BUCKET, prefix=f"{GCS_PROF_PREFIX}/"):
    size_mb = blob.size / 1024 / 1024
    print(f"  gs://{GCS_BUCKET}/{blob.name:<55} {size_mb:.2f} MB")

print(f"\n  Latest data : {new_latest_year} Round {new_latest_round}")
print(f"  Run date    : {datetime.date.today()}")
print(f"\n✅ Data update complete")

── Coverage (last 5 seasons) ─────────────────────────────────────────
  2026:  3 rounds,   66 result rows
  2025: 24 rounds,  479 result rows
  2024: 24 rounds,  479 result rows
  2023: 22 rounds,  440 result rows
  2022: 22 rounds,  440 result rows

── Final counts ──────────────────────────────────────────────────────
  races                             1,173
  results                          27,213
  qualifying                       11,036
  sprint_results                      502
  driver_standings                 35,427
  constructor_standings            13,664
  constructor_results              13,298
  drivers                             865
  constructors                        214
  circuits                             78
  seasons                              77
  mclaren_drivers                      55
  lap_times (unchanged)                 0  (Kaggle only)
  pit_stops (unchanged)                 0  (Kaggle only)

── Profile counts ────────────────────────────────────────